# Track 2 · 02 EEGNet — the organisers' classifier, on PROTEUS windows

**One job.** Train the organisers' EEGNet baseline on the warm-up data, check it learns,
score it the way the leaderboard does, and package it for submission.

Self-contained: upload this one file. The windows come from `track2_01_prepare` on Drive.

---

## The recipe, from source

Model and loss are `tracks/bci_decoding/solvers/eegnet.py`: `braindecode.models.EEGNet`,
cross-entropy, Adam at 1e-3. From the neuralbench `_motor_imagery_stream` config
(v0.4.0): **3 classes, label smoothing 0.1**, best validation balanced accuracy.

The warm-up contract is **streamed**: windows arrive one at a time (`B = 1`) in
chronological order within each participant x session, and each session runs on a fresh
copy of the model with `reset_state()` called first. Training is still ordinary batched
SGD; the streaming contract only binds the submission's `predict`.

## Scoring

**Balanced accuracy** per participant x session x interface (Graz / BrainHero) cell,
averaged equally. Three classes, chance is 33.3%. Our local split mirrors the stream
config's `SklearnSplit` (seed 33, 20% test / 20% valid). NeuralBench's public
references on this split: EEGNet 32.6%, CTNet 35.0%, REVE fine-tuned 36.0%.''')

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

EPOCHS = 35
KEEP = 'best'           # best val epoch (val people are a real 20% of 10 subjects)
LR = 1e-3
LR_SCHEDULE = 'cosine'
BATCH = 64
DROPOUT = 0.5
F1 = 16
LABEL_SMOOTHING = 0.0   # lower: task has only 1500 windows; smoothing hurts small-data runs
WEIGHT_DECAY = 0.0
N_SEEDS = 3
MODEL = 'eegnet'        # 'eegnet', 'reve_probe' (frozen), or 'reve_lora' (fine-tune)

RUN_CONTROL = True
MAKE_ZIP = True
SEED = 33

RUN_TAG = (f'proteus_{KEEP}_f{F1}_d{DROPOUT}_b{BATCH}_ls{LABEL_SMOOTHING}'
           f'_{LR_SCHEDULE}_x{N_SEEDS}')
print(f'NOTEBOOK VERSION: track2_02 v6 PROTEUS   model={MODEL}   settings {RUN_TAG}   epochs {EPOCHS}')

In [ ]:
%pip install -q 'braindecode>=1.8' huggingface_hub safetensors

import importlib
for mod in ('braindecode', 'torch', 'numpy', 'sklearn'):
    print(f'{mod:12} {importlib.import_module(mod).__version__}')

In [ ]:
import json, time, warnings, zipfile
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch
from sklearn.metrics import balanced_accuracy_score

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

BASE = Path(ROOT) / 'Track2'
DATA = BASE / 'dreyer2026proteus'
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def load_npz(path, tries=5):
    """np.load that survives the Drive mount dropping mid-read (Errno 107): remount,
    then retry."""
    for attempt in range(tries):
        try:
            with np.load(path) as z:
                return {k: z[k] for k in z.files}
        except (OSError, zipfile.BadZipFile, EOFError) as e:
            if attempt == tries - 1 or not ROOT.startswith('/content/drive'):
                raise
            print(f'Drive read failed on {Path(path).name} ({e}); remounting and retrying')
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)


man = json.loads((DATA / 'manifest.json').read_text())
recs = man['recordings']
group = {r['subject']: r['group'] for r in recs.values()}
parts = {g: ([], [], []) for g in ('train', 'val', 'test')}   # X, y, person
CH = None
for name, r in sorted(recs.items()):
    z = load_npz(DATA / 'windows' / r['file'])
    p = parts[group[r['subject']]]
    p[0].append(z['X'].astype('float32')); p[1].append(z['y'])
    p[2].append(np.full(len(z['y']), r['subject']))
    CH = CH or r['channels']
X, Y, P = {}, {}, {}
for g, (xs, ys, ps) in parts.items():
    X[g], Y[g], P[g] = np.concatenate(xs), np.concatenate(ys), np.concatenate(ps)
N_CHANS, N_TIMES = X['train'].shape[1:]
N_CLASSES = 3
SFREQ = man['preprocessing']['sfreq']
assert not set(P['test']) & set(P['train']) and not set(P['val']) & set(P['train'])
for g in ('train', 'val', 'test'):
    print(f'{g:5}  {len(set(P[g])):2d} people  {len(Y[g]):6,d} windows   '
          f'mi {(Y[g] == 0).sum():,} / sub {(Y[g] == 1).sum():,} / word {(Y[g] == 2).sum():,}')
print(f'{N_CHANS} channels, {N_TIMES} samples at {SFREQ:g} Hz, device {DEVICE}')


def scores(y, pred):
    return dict(bal_acc=float(balanced_accuracy_score(y, pred)), acc=float((y == pred).mean()))


def null(y, pred, n=200):
    """Balanced accuracy with the predictions shuffled across windows: same predictions,
    no link to the labels. A model has learned something only if it beats this."""
    return np.array([balanced_accuracy_score(y, rng.permutation(pred)) for _ in range(n)])

In [ ]:
from braindecode.models import EEGNet, REVE
import torch.nn.functional as F


def standardize(x):
    # Per-window per-channel standardization, matching the solver's _standardize().
    if not isinstance(x, torch.Tensor):
        x = torch.tensor(x)
    return (x - x.mean(dim=-1, keepdim=True)) / torch.sqrt(x.var(dim=-1, keepdim=True, unbiased=False) + 1e-8)


def train(Xtr, ytr, Xv, yv, epochs, label, quiet=False, seed=SEED):
    torch.manual_seed(seed)
    net = EEGNet(n_chans=N_CHANS, n_outputs=N_CLASSES, n_times=N_TIMES,
                 drop_prob=DROPOUT, F1=F1).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
             if LR_SCHEDULE == 'cosine' else None)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    Xtr_t, ytr_t, Xv_t = torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xv)
    gen = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def probs(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        return torch.cat([net(standardize(Xa[i:i + 512].to(DEVICE))).softmax(-1)
                          for i in range(0, len(Xa), 512)]).cpu().numpy()

    if not quiet:
        print(f'{label}: EEGNet {sum(p.numel() for p in net.parameters()):,} parameters, '
              f'{len(ytr):,} training windows, lr {LR_SCHEDULE}, seed {seed}')
    best, best_state, hist = -1.0, None, []
    for ep in range(1, epochs + 1):
        net.train()
        order = torch.randperm(len(ytr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(net(standardize(Xtr_t[b].to(DEVICE))), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        pv = probs(Xv_t)
        vl = float(-np.log(pv[np.arange(len(yv)), yv] + 1e-9).mean())
        sc = scores(yv, pv.argmax(1))
        hist.append(dict(epoch=ep, train_loss=tot / len(order), val_loss=vl, **sc))
        better = sc['bal_acc'] > best
        if better:
            best = sc['bal_acc']
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(order):.4f}  val loss {vl:.4f}  '
                  f'val bal-acc {sc["bal_acc"]:.4f}  lr {lr_now:.1e}{"  *" if better else ""}  '
                  f'{time.time() - t0:.0f}s')
    if KEEP == 'best':
        net.load_state_dict(best_state)
    return net, hist, probs




class ReveProbe(torch.nn.Module):
    """Frozen REVE encoder + a trainable linear head. No position bank needed: the
    head takes the pooled features and maps them to class logits."""

    def __init__(self, n_classes, device):
        super().__init__()
        self.net = REVE(n_outputs=n_classes, n_chans=N_CHANS, n_times=N_TIMES,
                        sfreq=SFREQ, chs_info=[{'ch_name': c} for c in CH])
        self.net.final_layer = torch.nn.Identity()
        self.head = torch.nn.Linear(self.net.embed_dim, n_classes)
        self.to(device).eval()
        for p_ in self.net.parameters():
            p_.requires_grad_(False)

    def forward(self, x):
        x = x.float()
        feats = self.net(x, return_features=True)['features']   # (B, ch, patch, embed)
        return self.head(feats.mean(dim=(1, 2)))


def train_reve(Xtr, ytr, Xv, yv, epochs, label, quiet=False, seed=SEED):
    torch.manual_seed(seed)
    net = ReveProbe(N_CLASSES, DEVICE)
    opt = torch.optim.Adam(net.head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
             if LR_SCHEDULE == 'cosine' else None)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    Xtr_t, ytr_t, Xv_t = torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xv)
    gen = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def probs(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        return torch.cat([net(standardize(Xa[i:i + 512].to(DEVICE))).softmax(-1)
                          for i in range(0, len(Xa), 512)]).cpu().numpy()

    if not quiet:
        print(f'{label}: REVE probe {sum(p.numel() for p in net.head.parameters()):,} '
              f'trainable head params (encoder frozen), {len(ytr):,} windows, '
              f'lr {LR_SCHEDULE}, seed {seed}')
    best, best_state, hist = -1.0, None, []
    for ep in range(1, epochs + 1):
        net.train()
        order = torch.randperm(len(ytr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(net(standardize(Xtr_t[b].to(DEVICE))), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        pv = probs(Xv_t)
        vl = float(-np.log(pv[np.arange(len(yv)), yv] + 1e-9).mean())
        sc = scores(yv, pv.argmax(1))
        hist.append(dict(epoch=ep, train_loss=tot / len(order), val_loss=vl, **sc))
        better = sc['bal_acc'] > best
        if better:
            best = sc['bal_acc']
            best_state = {k: v.detach().cpu().clone() for k, v in net.head.state_dict().items()}
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(order):.4f}  val loss {vl:.4f}  '
                  f'val bal-acc {sc["bal_acc"]:.4f}  lr {lr_now:.1e}{"  *" if better else ""}  '
                  f'{time.time() - t0:.0f}s')
    if KEEP == 'best':
        net.head.load_state_dict(best_state)
    return net, hist, probs




LORA_RANK = 8


class LoRALinear(torch.nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p_ in self.base.parameters():
            p_.requires_grad_(False)
        self.A = torch.nn.Parameter(torch.zeros(r, base.in_features))
        self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))
        torch.nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale


class ReveLoRA(torch.nn.Module):
    def __init__(self, n_classes, device, rank=LORA_RANK):
        super().__init__()
        self.net = REVE(n_outputs=n_classes, n_chans=N_CHANS, n_times=N_TIMES,
                        sfreq=SFREQ, chs_info=[{'ch_name': c} for c in CH])
        self.net.final_layer = torch.nn.Identity()
        self.head = torch.nn.Linear(self.net.embed_dim, n_classes)
        self.n_lora = 0
        for name, m in list(self.net.named_modules()):
            if isinstance(m, torch.nn.Linear) and name.split('.')[-1] in ('to_qkv', 'to_out'):
                parent = self.net.get_submodule(name.rsplit('.', 1)[0])
                setattr(parent, name.split('.')[-1], LoRALinear(m, rank, 2 * rank))
                self.n_lora += 1
        for p_ in self.net.parameters():
            p_.requires_grad_(False)
        for n_, p_ in self.net.named_parameters():
            if n_.endswith(('.A', '.B')):
                p_.requires_grad_(True)
        self.to(device).eval()

    def forward(self, x):
        x = x.float()
        feats = self.net(x, return_features=True)['features']
        return self.head(feats.mean(dim=(1, 2)))

    def trainable_params(self):
        return ([p_ for n_, p_ in self.net.named_parameters() if p_.requires_grad]
                + list(self.head.parameters()))


def train_reve_lora(Xtr, ytr, Xv, yv, epochs, label, quiet=False, seed=SEED):
    torch.manual_seed(seed)
    net = ReveLoRA(N_CLASSES, DEVICE)
    opt = torch.optim.Adam(net.trainable_params(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
             if LR_SCHEDULE == 'cosine' else None)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    Xtr_t, ytr_t, Xv_t = torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xv)
    gen = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def probs(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        return torch.cat([net(standardize(Xa[i:i + 512].to(DEVICE))).softmax(-1)
                          for i in range(0, len(Xa), 512)]).cpu().numpy()

    if not quiet:
        n_ad = sum(p_.numel() for n_, p_ in net.net.named_parameters() if p_.requires_grad)
        print(f'{label}: REVE LoRA {n_ad:,} adapter params, {net.n_lora} adapters, '
              f'{len(ytr):,} windows, lr {LR_SCHEDULE}, seed {seed}')
    best, best_state, hist = -1.0, None, []
    for ep in range(1, epochs + 1):
        net.train()
        order = torch.randperm(len(ytr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(net(standardize(Xtr_t[b].to(DEVICE))), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        pv = probs(Xv_t)
        vl = float(-np.log(pv[np.arange(len(yv)), yv] + 1e-9).mean())
        sc = scores(yv, pv.argmax(1))
        hist.append(dict(epoch=ep, train_loss=tot / len(order), val_loss=vl, **sc))
        better = sc['bal_acc'] > best
        if better:
            best = sc['bal_acc']
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()
                          if k.endswith(('.A', '.B')) or k.startswith('head.')}
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(order):.4f}  val loss {vl:.4f}  '
                  f'val bal-acc {sc["bal_acc"]:.4f}  lr {lr_now:.1e}{"  *" if better else ""}  '
                  f'{time.time() - t0:.0f}s')
    if KEEP == 'best':
        net.load_state_dict(best_state, strict=False)
    return net, hist, probs


print('ready')

## Step 1 — positive control

A pattern whose sign depends on the label is planted into a copy of the windows and the
same loop runs on it. If this does not learn, the loop is broken and Step 2 means nothing.

In [ ]:
if RUN_CONTROL:
    t_ax = np.arange(N_TIMES) / SFREQ
    pattern = (rng.standard_normal((N_CHANS, 1)) * np.sin(2 * np.pi * 10.0 * t_ax)).astype('float32')

    def plant(Xa, ya):
        return (Xa + 0.5 * (2 * ya - 1)[:, None, None] * pattern[None]).astype('float32')

    sub = rng.choice(len(Y['train']), min(4000, len(Y['train'])), replace=False)
    _, hist_c, _ = train(plant(X['train'][sub], Y['train'][sub]), Y['train'][sub],
                         plant(X['val'], Y['val']), Y['val'], 10, 'control', quiet=True)
    best_c = max(h['bal_acc'] for h in hist_c)
    print(f'control val balanced accuracy: epoch 1 {hist_c[0]["bal_acc"]:.3f} -> best {best_c:.3f}')
    print('CONTROL PASSES: the loop learns a planted signal' if best_c > 0.9 else
          'CONTROL FAILS: the loop cannot learn a planted signal; fix before Step 2')
else:
    print('control skipped')

## Step 2 — the real run

What to look for is validation balanced accuracy rising above its own null (the same
predictions shuffled across windows) and the best epoch coming after the first.

In [ ]:
members = []
for i in range(N_SEEDS):
    train_fn = {'reve_probe': train_reve, 'reve_lora': train_reve_lora}.get(MODEL, train)
    net_i, hist_i, probs_i = train_fn(X['train'], Y['train'], X['val'], Y['val'], EPOCHS,
                                      f'real, seed {i + 1}/{N_SEEDS}', seed=SEED + i)
    best_i = max(hist_i, key=lambda h: h['bal_acc']) if KEEP == 'best' else hist_i[-1]
    members.append(dict(net=net_i, hist=hist_i, probs=probs_i, best=best_i,
                        test=scores(Y['test'], probs_i(X['test']).argmax(1))))
    print(f'  -> seed {i + 1}: kept epoch {best_i["epoch"]} ({KEEP}), val bal-acc '
          f'{best_i["bal_acc"]:.4f}, test bal-acc {members[-1]["test"]["bal_acc"]:.4f}\\n')


def predict(Xa):
    return np.mean([m['probs'](Xa) for m in members], axis=0).argmax(1)


hist = members[0]['hist']
best = max(hist, key=lambda h: h['bal_acc'])
pv = predict(X['val'])
sv = scores(Y['val'], pv)
nullv = null(Y['val'], pv)
thr = float(np.percentile(nullv, 95))
climb = best['bal_acc'] - hist[0]['bal_acc']
print(f'averaged val bal-acc {sv["bal_acc"]:.4f} against shuffled predictions: mean '
      f'{nullv.mean():.4f}  sd {nullv.std():.4f}  95th pct {thr:.4f}')
TRENDING = bool(sv['bal_acc'] > thr and best['epoch'] > 1 and climb > 2 * nullv.std())
print('Trending: validation improved past epoch 1 and beats its null.' if TRENDING else
      'Above the null but no clear improvement after epoch 1.' if sv['bal_acc'] > thr else
      'Not trending: validation never beats its own null.')

pt = predict(X['test'])
st = scores(Y['test'], pt)
per_person = {str(s): float((pt[P['test'] == s] == Y['test'][P['test'] == s]).mean())
              for s in np.unique(P['test'])}
pp = np.array(list(per_person.values()))
seed_b = [m['test']['bal_acc'] for m in members]
print(f'\\ntest (the SklearnSplit test people)')
print(f'  EEGNet, mean of {N_SEEDS} seeds   balanced accuracy {st["bal_acc"]:.4f}   '
      f'accuracy {st["acc"]:.4f}   (chance 0.333)')
print(f'  single seeds: {", ".join(f"{b:.4f}" for b in seed_b)}   '
      f'(spread {max(seed_b) - min(seed_b):.4f})')
print(f'  per person: min {pp.min():.2f}  median {np.median(pp):.2f}  max {pp.max():.2f}')

for i, m in enumerate(members):
    torch.save(m['net'].state_dict(), WEIGHTS / f'bci_eegnet_{RUN_TAG}_s{i}.pt')
(WEIGHTS / f'bci_eegnet_{RUN_TAG}.json').write_text(json.dumps(dict(
    run_tag=RUN_TAG, train_people=sorted(str(s) for s in set(P['train'])),
    check_people=sorted(str(s) for s in set(P['val'])), keep=KEEP, f1=F1, dropout=DROPOUT,
    batch=BATCH, label_smoothing=LABEL_SMOOTHING, lr_schedule=LR_SCHEDULE,
    n_seeds=N_SEEDS, test=st, per_person=per_person,
    members=[dict(best_epoch=m['best']['epoch'], test=m['test'], history=m['hist'])
             for m in members]), indent=1))
print(f'\\nsaved {N_SEEDS} members -> {WEIGHTS / f"bci_eegnet_{RUN_TAG}_s*.pt"}')

## Step 3 — package for submission

The zip is one weights file per seed, the solver and a small config. The solver matches
incoming electrodes to the training ones **by name** and prints how many matched into the
Codabench log. Before zipping, it is loaded back the way the harness loads it and checked:
its predictions must equal this notebook's, and reordering or adding electrodes must not
change them.

In [ ]:
# Source of truth is solvers/track2_bci_eegnet.py in the repo.
SUBMISSION_PY = '"""Track 02 submission: EEGNet classifying cued mental commands.\n\nThe model follows the organisers\' `eegnet.py` baseline (braindecode EEGNet,\ncross-entropy). The weights come from `track2_02_eegnet.ipynb`, trained on\nPROTEUS (NEMAR nm000290) windows made with the harness\'s own preprocessing.\nA zip may hold several copies trained from different seeds; their class\nprobabilities are averaged before the argmax.\n\nThe streaming contract (warm-up and sealed): windows reach `predict` one at a\ntime (`B = 1`) in chronological order within each participant x session, and\neach session runs on a fresh `copy.deepcopy` of this model after calling its\noptional `reset_state()`. `reset_state` clears the optional running\nper-channel mean/variance kept by the per-window standardizer, which is the\nonly state this submission keeps.\n\nInputs are matched to the training electrodes by name: extra electrodes are\nignored, missing ones are filled with zeros, and it raises if none match. The\nclass count is fixed by training (three on the warm-up), so it also raises if\nthe harness asks for a different number of classes.\n"""\n\nimport json\n\nimport torch\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import EEGNet\n\n\ndef resample_fft(x, n_out):\n    """Resample along time via the rfft spectrum. Pure torch, since nothing is\n    installed at evaluation."""\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass ChannelMatchedClassifier:\n\n    def __init__(self, cfg, in_names, in_sfreq, device):\n        self.device = device\n        self.n_times = int(cfg["n_times"])\n        self.sfreq = float(cfg["sfreq"])\n        self.in_sfreq = float(in_sfreq)\n        self.nets = [\n            EEGNet(\n                n_chans=len(cfg["channels"]), n_outputs=int(cfg["n_classes"]),\n                n_times=self.n_times, drop_prob=float(cfg["dropout"]),\n                F1=int(cfg.get("f1", 8)),\n            ).to(device).eval()\n            for _ in cfg["weight_files"]\n        ]\n\n        lookup = {name.casefold(): i for i, name in enumerate(in_names)}\n        self.src = [lookup.get(name.casefold(), -1) for name in cfg["channels"]]\n        missing = [n for n, i in zip(cfg["channels"], self.src) if i < 0]\n        print(\n            f"[EEGNet-BCI] matched {len(self.src) - len(missing)}/{len(self.src)} "\n            f"training channels by name among {len(in_names)} inputs; "\n            f"zero-filled: {missing if missing else \'none\'}"\n        )\n        if len(missing) == len(self.src):\n            raise RuntimeError(\n                f"no training channel found in the input: {list(in_names)[:8]}"\n            )\n        # Running per-channel mean/variance over the current session, reset by\n        # reset_state(). Only set when the harness reports a channel count we\n        # can standardize (it always does: meta["n_chans"]).\n        self.adapt_mean = None\n        self.adapt_var = None\n        self.adapt_count = 0\n\n    def reset_state(self):\n        self.adapt_mean = None\n        self.adapt_var = None\n        self.adapt_count = 0\n\n    def _standardize(self, x):\n        # Per-window per-channel standardization, the training-time pipeline.\n        mean = x.mean(dim=-1, keepdim=True)\n        var = x.var(dim=-1, keepdim=True, unbiased=False)\n        return (x - mean) / torch.sqrt(var + 1e-8)\n\n    @torch.no_grad()\n    def predict(self, X):\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        # Match the training rate and window length. At the harness\'s own\n        # settings both are no-ops.\n        if self.in_sfreq != self.sfreq:\n            out = resample_fft(\n                out, int(round(out.shape[-1] * self.sfreq / self.in_sfreq))\n            )\n        if out.shape[-1] > self.n_times:\n            out = out[..., : self.n_times]\n        elif out.shape[-1] < self.n_times:\n            out = F.pad(out, (0, self.n_times - out.shape[-1]))\n        out = self._standardize(out)\n        probs = torch.stack([net(out).softmax(-1) for net in self.nets]).mean(0)\n        return probs.argmax(-1)\n\n\nclass Solver(CompetSolver):\n\n    name = "EEGNet-BCI"\n\n    requirements = ["pip::braindecode"]\n\n    def load_model(self, meta):\n        sub = meta["submission_dir"]\n        cfg = json.loads((sub / "bci_eegnet_config.json").read_text())\n        if meta.get("n_classes") not in (None, cfg["n_classes"]):\n            raise RuntimeError(\n                f"trained for {cfg[\'n_classes\']} classes, harness asks for "\n                f"{meta[\'n_classes\']}"\n            )\n        in_names = meta.get("ch_names") or [c["ch_name"] for c in meta["chs_info"]]\n        model = ChannelMatchedClassifier(cfg, in_names, meta["sfreq"], meta["device"])\n        for net, name in zip(model.nets, cfg["weight_files"]):\n            state = torch.load(\n                sub / name, map_location=meta["device"], weights_only=True\n            )\n            net.load_state_dict(state)\n        return model\n'


In [ ]:
if MAKE_ZIP and not (TRENDING or globals().get('FORCE_ZIP')):
    print('Not packaging: the run did not clear its null. Set FORCE_ZIP = True to override.')
elif MAKE_ZIP:
    import importlib.util, shutil, sys, types
    out = BASE / 'submissions' / f'track2_eegnet_{RUN_TAG}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    files = [f'weights_{i}.pt' for i in range(len(members))]
    for f_, m in zip(files, members):
        torch.save({k: v.cpu() for k, v in m['net'].state_dict().items()}, out / f_)
    (out / 'bci_eegnet_config.json').write_text(json.dumps(dict(
        channels=list(CH), n_times=int(N_TIMES), sfreq=SFREQ, n_classes=N_CLASSES,
        dropout=DROPOUT, f1=F1, weight_files=files,
        trained_on=f'Dreyer 2023, {len(set(P["train"]))} people (61-81 excluded), harness default preprocessing',
        test_bal_acc=st['bal_acc']), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)

    # Load it the way the harness does, with a stand-in for the base class.
    stub = types.ModuleType('benchmark_utils.base_solver')
    stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = stub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def run(names, Xa, n_classes=N_CLASSES):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None,
                                     'sfreq': SFREQ, 'submission_dir': out,
                                     'device': DEVICE, 'n_classes': n_classes})
        return m.predict(torch.tensor(Xa)).cpu().numpy()

    k = rng.choice(len(Y['test']), min(256, len(Y['test'])), replace=False)
    Xs = X['test'][k]
    base = run(CH, Xs)
    perm = rng.permutation(len(CH))
    shuffled = run([CH[i] for i in perm], Xs[:, perm])
    wide = run(list(CH) + ['EOG1', 'EMG1'],
               np.concatenate([Xs, rng.standard_normal((len(Xs), 2, N_TIMES)).astype('float32')], 1))
    keep_ch = [c for c in CH if c != 'Cz']
    part = run(keep_ch, Xs[:, [CH.index(c) for c in keep_ch]])
    try:
        run(CH, Xs[:4], n_classes=4)
        wrong_cls = False
    except RuntimeError:
        wrong_cls = True

    checks = [
        ('shape and labels', base.shape == (len(Xs),) and set(np.unique(base)) <= {0, 1, 2},
         f'{base.shape}, classes {sorted(set(base.tolist()))}'),
        ('same as this notebook', np.array_equal(base, pt[k]),
         f'{int((base != pt[k]).sum())} of {len(k)} differ'),
        ('order-independent', np.array_equal(base, shuffled), 'electrodes shuffled'),
        ('extra electrodes ignored', np.array_equal(base, wide), 'two unknown channels added'),
        ('missing electrode', part.shape == (len(Xs),), 'Cz zero-filled, still runs'),
        ('wrong class count raises', wrong_cls, 'asked for 4 classes: refuses rather than guessing'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e3:.0f} KB)')
        print('Download it from Drive and upload to Codabench (Track 2).')
    else:
        print('\nNot zipping: a check failed.')